# Кандидатогенерация для категории услуг Avito

Тестовое задание для отбора на Data Science Bootcamp от Авито

Нужно для каждого поискового запроса вернуть не более 50 объявлений из
benchmark-корпуса. Метрика: Recall@50. Это важно: если подходящее объявление
не попало в первые 50 кандидатов, последующий ранжировщик уже не сможет его
восстановить

Я начал с простого TF-IDF по заголовкам, а затем постепенно добавлял новые
источники кандидатов. В ноутбуке оставил не только финальный код, но и логику,
по которой пришёл к итоговому решению

## 1. План работы

Мой план был таким:

1. разобраться со структурой данных и тем, насколько train похож на benchmark
2. собрать простой текстовый baseline
3. проверить, помогают ли описания, опечатки, география и история запросов
4. объединить разные списки кандидатов
5. отдельно проверить формат итогового CSV

Поскольку порядок внутри топ-50 не влияет на метрику, я не обучал полноценный
ранжировщик. Вместо этого старался получить разные по смыслу наборы кандидатов
и аккуратно объединить их

In [1]:
from __future__ import annotations

import os
import re
import time
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
TOP_K = 50
RETRIEVAL_K = 150
VALIDATION_SIZE = 600
RRF_CONSTANT = 30


def find_data_dir() -> Path:
    """Находит датасет без привязки основного кода к конкретному компьютеру."""
    required = {
        "train.parquet",
        "benchmark_queries.parquet",
        "benchmark_items.parquet",
    }
    candidates = [
        Path(os.environ.get("AVITO_DATA_DIR", "data")),
        Path.home() / "Downloads" / "dataset",
        Path("../data"),
    ]
    for candidate in candidates:
        if candidate.is_dir() and required.issubset({p.name for p in candidate.iterdir()}):
            return candidate.resolve()
    raise FileNotFoundError(
        "Не найдены parquet-файлы. Положите их в data/ или задайте AVITO_DATA_DIR."
    )


DATA_DIR = find_data_dir()
OUTPUT_PATH = Path("answer.csv")
print(f"Каталог данных найден: {DATA_DIR.name}")

Каталог данных найден: dataset


## 2. Загрузка данных

Идентификаторы сразу оставляю строками, чтобы не потерять ведущие нули. Из
`train` загружаю только признаки, которые понадобятся дальше: полный файл
заметно больше остальных и лишние столбцы увеличивают расход памяти

In [2]:
QUERY_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
TRAIN_COLUMNS = QUERY_COLUMNS + [
    "item_id",
    "item_microcat_id",
    "item_location_id",
]

train = pd.read_parquet(DATA_DIR / "train.parquet", columns=TRAIN_COLUMNS)
benchmark_queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
benchmark_items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

train["item_id"] = train["item_id"].astype(str)
benchmark_queries["query_id"] = benchmark_queries["query_id"].astype(str)
benchmark_items["item_id"] = benchmark_items["item_id"].astype(str)

assert benchmark_queries["query_id"].is_unique
assert benchmark_items["item_id"].is_unique

summary = pd.DataFrame(
    {
        "rows": [len(train), len(benchmark_queries), len(benchmark_items)],
        "unique_ids": [
            train["item_id"].nunique(),
            benchmark_queries["query_id"].nunique(),
            benchmark_items["item_id"].nunique(),
        ],
    },
    index=["train", "benchmark_queries", "benchmark_items"],
)
display(summary)

,rows,unique_ids
train,497673,344825
benchmark_queries,2452,2452
benchmark_items,189212,189212


In [3]:
def normalize_text(series: pd.Series) -> pd.Series:
    """Лёгкая детерминированная нормализация без внешних NLP-моделей."""
    return (
        series.fillna("")
        .astype(str)
        .str.lower()
        .str.replace("ё", "е", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


for frame in (train, benchmark_queries):
    frame["_query_norm"] = normalize_text(frame["search_query"])

benchmark_item_ids = benchmark_items["item_id"].to_numpy()
benchmark_item_id_set = set(benchmark_item_ids)
item_index = {item_id: i for i, item_id in enumerate(benchmark_item_ids)}

eda = {
    "train_rows": len(train),
    "train_unique_queries": train["_query_norm"].nunique(),
    "benchmark_query_count": len(benchmark_queries),
    "benchmark_item_count": len(benchmark_items),
    "train_items_in_benchmark": train.loc[
        train["item_id"].isin(benchmark_item_id_set), "item_id"
    ].nunique(),
    "benchmark_queries_seen_in_train_share": benchmark_queries[
        "_query_norm"
    ].isin(set(train["_query_norm"])).mean(),
    "positive_exact_location_share": (
        train["search_location_id"] == train["item_location_id"]
    ).mean(),
}
display(pd.Series(eda, name="value").to_frame())

,value
train_rows,497673.000000
train_unique_queries,73874.000000
benchmark_query_count,2452.000000
benchmark_item_count,189212.000000
train_items_in_benchmark,18142.000000
benchmark_queries_seen_in_train_share,0.373573
positive_exact_location_share,0.831020


## 3. Что лежит в данных

Сначала посмотрю на размеры файлов, типы столбцов и несколько живых примеров
`train` представляет собой не таблицу объявлений, а положительные пары
«запрос, выбранное объявление». Один и тот же запрос или объект может
встречаться много раз

In [4]:
file_overview = []
for file_name in ["train.parquet", "benchmark_queries.parquet", "benchmark_items.parquet"]:
    path = DATA_DIR / file_name
    parquet_file = pq.ParquetFile(path)
    file_overview.append(
        {
            "file": file_name,
            "rows": parquet_file.metadata.num_rows,
            "columns": len(parquet_file.schema_arrow),
            "size_mb": round(path.stat().st_size / 1024**2, 1),
        }
    )

display(pd.DataFrame(file_overview))

schema = pd.DataFrame(
    {
        "column": [field.name for field in pq.ParquetFile(DATA_DIR / "train.parquet").schema_arrow],
        "dtype": [str(field.type) for field in pq.ParquetFile(DATA_DIR / "train.parquet").schema_arrow],
    }
)
display(schema)

,file,rows,columns,size_mb
0,train.parquet,497673,19,467.5
1,benchmark_queries.parquet,2452,6,0.1
2,benchmark_items.parquet,189212,14,186.6


,column,dtype
0,search_query,string
1,search_location_id,int64
2,search_is_delivery_search,int32
3,search_infm_params_text,string
4,search_category,int64
5,item_title_raw,string
6,item_rating_reviews_count,double
7,item_rating,double
8,item_price,"decimal128(27, 15)"
9,item_microcat_id,int64


In [5]:
# Обрезаю длинные поля только для удобного вывода, исходные данные не меняются.
query_examples = benchmark_queries[
    ["search_query", "search_location_id", "search_infm_params_text"]
].head(8)

item_examples = benchmark_items[
    ["item_id", "item_title_raw", "item_infm_params_text", "item_location_id"]
].head(5).copy()
item_examples["item_infm_params_text"] = (
    item_examples["item_infm_params_text"].fillna("").str.slice(0, 100) + "…"
)

display(query_examples)
display(item_examples)

,search_query,search_location_id,search_infm_params_text
0,перевозки владикавказ тбилиси,649820,
1,обзвон по базе,107620,Вид услуги Деловые услуги
2,липоредукция подбородка,637640,"Вид услуги Красота, здоровье"
3,подъемник 4 х стоечный,662810,
4,монтаж видеодомофонов,642790,
5,боевое самбо,659880,"Тип услуги Дизайн, рисование Вид услуги Обучен..."
6,услуги бухгалтера для ип,633080,Вид услуги Деловые услуги Тип услуги Бухгалтер...
7,бригада по установке забора из профнастила,631870,


,item_id,item_title_raw,item_infm_params_text,item_location_id
0,111eb8b979577d79,Ремонт/выкуп компьют. и ноутбуков с выездом на...,Вид услуги Компьютерная помощь Место оказания ...,631060
1,75fc8e10f5a66fc4,Обучение ребенка чтению,"Вид услуги Обучение, курсы Место оказания услу...",631870
2,3f6ae81704565b9c,Афрокудри 5+,"Вид услуги Красота, здоровье Место оказания ус...",628500
3,0618dec37a59a21a,Монтаж малых архитектурных форм,Вид услуги Строительство Место оказания услуг ...,637640
4,13da2c81574677ed,Репетитор по истории 10 класс ЕГЭ,"Вид услуги Обучение, курсы Место оказания услу...",624850


### Первые наблюдения

- запросы очень короткие: чаще всего 2-4 слова
- заголовок объявления тоже короткий, а полезные детали могут быть только в
  параметрах или описании
- фильтры представлены обычным текстом и иногда противоречат самому запросу
- почти весь train относится к категории услуг (`114`), но в корпусе есть и
  небольшое число объектов других категорий
- часть item_id из train присутствует в benchmark-корпусе, поэтому историю
  взаимодействий можно использовать как один из источников, но не как
  единственный способ поиска

In [6]:
query_word_count = benchmark_queries["_query_norm"].str.split().str.len()
train_query_frequency = train["_query_norm"].value_counts()
microcat_counts = benchmark_items["item_microcat_id"].value_counts().head(10)

In [7]:
data_diagnostics = pd.Series(
    {
        "median_query_words": float(query_word_count.median()),
        "benchmark_unique_microcats": int(benchmark_items["item_microcat_id"].nunique()),
        "largest_microcat_share": float(microcat_counts.iloc[0] / len(benchmark_items)),
        "queries_with_empty_filters_share": float(
            benchmark_queries["search_infm_params_text"].fillna("").eq("").mean()
        ),
        "train_query_seen_once_share": float(train_query_frequency.eq(1).mean()),
        "train_item_overlap_with_benchmark": int(eda["train_items_in_benchmark"]),
        "benchmark_query_seen_in_train_share": float(
            eda["benchmark_queries_seen_in_train_share"]
        ),
        "benchmark_locations_without_exact_items": int(
            benchmark_queries["search_location_id"]
            .map(benchmark_items["item_location_id"].value_counts())
            .fillna(0)
            .eq(0)
            .sum()
        ),
    },
    name="value",
)
display(data_diagnostics.to_frame())

,value
median_query_words,3.000000
benchmark_unique_microcats,752.000000
largest_microcat_share,0.033587
queries_with_empty_filters_share,0.630506
train_query_seen_once_share,0.594553
train_item_overlap_with_benchmark,18142.000000
benchmark_query_seen_in_train_share,0.373573
benchmark_locations_without_exact_items,427.000000


Главный вывод из EDA: нельзя просто отфильтровать корпус по точному
`location_id`. Хотя совпадение есть в 83% положительных пар train, для части
benchmark-запросов в корпусе вообще нет объявлений с таким идентификатором.
Поэтому дальше география используется как бонус, а не как обязательное
условие

Ещё один вывод: около 37% текстов benchmark-запросов уже встречались в train.
Это делает историю полезной, но для большинства запросов всё равно нужен
полноценный текстовый поиск

### Какие гипотезы я проверял

1. **TF-IDF только по заголовкам.** Быстро, но часто не видит узкую услугу,
   которая упомянута лишь в описании
2. **Добавить параметры и описание.** Дало заметный прирост относительно
   заголовков
3. **Добавить текст фильтров к запросу.** В TF-IDF общие слова «вид услуги»
   и «тип услуги» размывали короткий запрос. В BM25 фильтры дали небольшой
   прирост, поэтому использую их только в этом источнике
4. **Жёстко оставить только точную локацию.** На локальной проверке это часто
   помогало, но EDA показал, что так можно потерять примерно 17% релевантных
   пар. В финале заменил фильтр мягким бонусом
5. **Использовать похожие запросы из train.** Этот источник оказался полезен
   для синонимов и объявлений с неочевидными заголовками

## 4. Локальная валидация без утечки

В benchmark нет разметки. Для локальной проверки используются строки train,
чьи `item_id` присутствуют в `benchmark_items`. Получается задача поиска по
тому же корпусу, что и в финальном прогнозе

Для валидации выбираются редкие запросы (не более 20 появлений в train), а
все строки с их текстами полностью удаляются из данных для построения
поведенческих источников. Тексты самих объявлений остаются в корпусе, как и
должны оставаться в реальной задаче retrieval

In [8]:
overlap = train[train["item_id"].isin(benchmark_item_id_set)].copy()
query_frequency = train["_query_norm"].value_counts()

validation_groups = (
    overlap.groupby(QUERY_COLUMNS + ["_query_norm"], dropna=False)["item_id"]
    .agg(lambda values: frozenset(values))
    .reset_index(name="relevant_items")
)
validation_pool = validation_groups[
    validation_groups["_query_norm"].map(query_frequency).le(20)
]
validation = validation_pool.sample(
    n=min(VALIDATION_SIZE, len(validation_pool)),
    random_state=SEED,
).reset_index(drop=True)

validation_query_texts = set(validation["_query_norm"])
prior_train_validation = train[
    ~train["_query_norm"].isin(validation_query_texts)
].copy()

print(f"Validation-запросов: {len(validation):,}")
print(
    "Среднее число релевантных объявлений:",
    round(validation["relevant_items"].map(len).mean(), 3),
)

Validation-запросов: 600
Среднее число релевантных объявлений: 1.027


## 5. Текстовый поиск

Матрица сходств вычисляется батчами. Полная матрица
`число запросов × число объявлений` в памяти не хранится

In [9]:
all_query_frames = pd.concat(
    [validation[QUERY_COLUMNS + ["_query_norm"]], benchmark_queries[QUERY_COLUMNS + ["_query_norm"]]],
    ignore_index=True,
)
all_query_texts = all_query_frames["_query_norm"]
bm25_query_texts = normalize_text(
    all_query_texts + " " + all_query_frames["search_infm_params_text"].fillna("")
)

item_titles = normalize_text(benchmark_items["item_title_raw"])
item_title_params = normalize_text(
    benchmark_items["item_title_raw"].fillna("")
    + " "
    + benchmark_items["item_infm_params_text"].fillna("")
)
item_full_text = normalize_text(
    benchmark_items["item_title_raw"].fillna("")
    + " "
    + benchmark_items["item_title_raw"].fillna("")
    + " "
    + benchmark_items["item_infm_params_text"].fillna("")
    + " "
    + benchmark_items["item_description_raw"].fillna("").str.slice(0, 2500)
)


def retrieve_top_indices(
    vectorizer: TfidfVectorizer,
    documents: pd.Series,
    queries: pd.Series,
    top_n: int = RETRIEVAL_K,
    batch_size: int = 32,
) -> tuple[np.ndarray, dict[str, float]]:
    """Строит TF-IDF индекс и возвращает top-N индексов для каждого запроса."""
    started = time.perf_counter()
    document_matrix = vectorizer.fit_transform(documents)
    query_matrix = vectorizer.transform(queries)
    result: list[np.ndarray] = []

    for start in range(0, len(queries), batch_size):
        scores = (query_matrix[start : start + batch_size] @ document_matrix.T).toarray()
        top = np.argpartition(scores, -top_n, axis=1)[:, -top_n:]
        rows = np.arange(len(top))[:, None]
        order = np.argsort(scores[rows, top], axis=1)[:, ::-1]
        result.extend(top[rows, order])

    stats = {
        "features": float(document_matrix.shape[1]),
        "non_zero": float(document_matrix.nnz),
        "seconds": round(time.perf_counter() - started, 1),
    }
    return np.asarray(result, dtype=np.int32), stats


def retrieve_bm25_top_indices(
    documents: pd.Series,
    queries: pd.Series,
    top_n: int = RETRIEVAL_K,
    batch_size: int = 32,
) -> tuple[np.ndarray, dict[str, float]]:
    """Возвращает top-N по BM25 без внешних поисковых сервисов."""
    started = time.perf_counter()
    vectorizer = CountVectorizer(
        ngram_range=(1, 1),
        min_df=2,
        max_features=250_000,
        token_pattern=r"(?u)\b\w\w+\b",
        dtype=np.float32,
    )
    document_matrix = vectorizer.fit_transform(documents).tocsr()
    document_length = np.asarray(document_matrix.sum(axis=1)).ravel()
    average_length = document_length.mean()
    document_frequency = np.diff(document_matrix.tocsc().indptr)
    inverse_frequency = np.log1p(
        (len(documents) - document_frequency + 0.5)
        / (document_frequency + 0.5)
    ).astype(np.float32)

    k1, b = 1.2, 0.75
    row_numbers = np.repeat(
        np.arange(document_matrix.shape[0]),
        np.diff(document_matrix.indptr),
    )
    term_frequency = document_matrix.data
    denominator = term_frequency + k1 * (
        1 - b + b * document_length[row_numbers] / average_length
    )
    document_matrix.data = (
        term_frequency * (k1 + 1) / denominator * inverse_frequency[document_matrix.indices]
    ).astype(np.float32)

    query_matrix = vectorizer.transform(queries).tocsr()
    query_matrix.data[:] = 1.0
    result: list[np.ndarray] = []
    for start in range(0, len(queries), batch_size):
        scores = (
            query_matrix[start : start + batch_size] @ document_matrix.T
        ).toarray()
        top = np.argpartition(scores, -top_n, axis=1)[:, -top_n:]
        rows = np.arange(len(top))[:, None]
        order = np.argsort(scores[rows, top], axis=1)[:, ::-1]
        result.extend(top[rows, order])

    stats = {
        "features": float(document_matrix.shape[1]),
        "non_zero": float(document_matrix.nnz),
        "seconds": round(time.perf_counter() - started, 1),
    }
    return np.asarray(result, dtype=np.int32), stats


char_candidates, char_stats = retrieve_top_indices(
    TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 5),
        min_df=2,
        max_features=150_000,
        sublinear_tf=True,
        dtype=np.float32,
    ),
    item_titles,
    all_query_texts,
)

title_word_candidates, title_word_stats = retrieve_top_indices(
    TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=120_000,
        sublinear_tf=True,
        token_pattern=r"(?u)\b\w\w+\b",
        dtype=np.float32,
    ),
    item_titles,
    all_query_texts,
)

full_word_candidates, full_word_stats = retrieve_top_indices(
    TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=250_000,
        sublinear_tf=True,
        token_pattern=r"(?u)\b\w\w+\b",
        dtype=np.float32,
    ),
    item_full_text,
    all_query_texts,
)

bm25_candidates, bm25_stats = retrieve_bm25_top_indices(
    item_full_text,
    bm25_query_texts,
)

bm25_params_candidates, bm25_params_stats = retrieve_bm25_top_indices(
    item_title_params,
    bm25_query_texts,
)

display(
    pd.DataFrame(
        [
            char_stats,
            title_word_stats,
            full_word_stats,
            bm25_stats,
            bm25_params_stats,
        ],
        index=[
            "char_title",
            "word_title",
            "word_full",
            "bm25_full",
            "bm25_title_params",
        ],
    )
)

,features,non_zero,seconds
char_title,150000.0,19861691.0,22.3
word_title,71178.0,1265666.0,2.5
word_full,250000.0,57027748.0,80.6
bm25_full,250000.0,30376425.0,30.3
bm25_title_params,59654.0,12346799.0,12.0


## 6. Похожие запросы, категории и география

Поиск похожих запросов строится только на train. Для каждого из 50 ближайших
запросов агрегируются ранее выбранные объявления из benchmark-корпуса и
микрокатегории. Отдельно оцениваются совместимые локации: восемь наиболее
частых `item_location_id` для каждого `search_location_id`

In [10]:
item_locations = benchmark_items["item_location_id"].to_numpy()
item_microcats = benchmark_items["item_microcat_id"].to_numpy()
item_categories = benchmark_items["item_category_id"].to_numpy()

ratings = benchmark_items["item_rating"].fillna(0).to_numpy(dtype=np.float32)
review_signal = np.log1p(
    benchmark_items["item_rating_reviews_count"].fillna(0).to_numpy(dtype=np.float32)
)
item_quality = (ratings / 5.0) * np.minimum(review_signal / 5.0, 1.0)


def build_behavioral_sources(
    history: pd.DataFrame,
    targets: pd.DataFrame,
    nearest_queries: int = 50,
) -> dict[str, object]:
    """Создаёт query-neighbor, history, popularity, microcat и geo priors."""
    query_table = (
        history.groupby("_query_norm")
        .agg(
            dominant_microcat=(
                "item_microcat_id",
                lambda values: values.value_counts().index[0],
            )
        )
        .reset_index()
    )

    history_in_corpus = history[history["item_id"].isin(benchmark_item_id_set)]
    history_counts = history_in_corpus.groupby(["_query_norm", "item_id"]).size()

    item_history: dict[str, np.ndarray] = {}
    for query_text, counts in history_counts.groupby(level=0):
        ranked_ids = counts.sort_values(ascending=False).index.get_level_values(1)[:TOP_K]
        item_history[query_text] = np.asarray(
            [item_index[item_id] for item_id in ranked_ids],
            dtype=np.int32,
        )

    location_history_counts = history_in_corpus.groupby(
        ["_query_norm", "search_location_id", "item_id"]
    ).size()
    item_history_by_location: dict[tuple[str, int], np.ndarray] = {}
    for (query_text, search_location), counts in location_history_counts.groupby(
        level=[0, 1]
    ):
        ranked_ids = counts.sort_values(ascending=False).index.get_level_values(2)[:TOP_K]
        item_history_by_location[(str(query_text), int(search_location))] = np.asarray(
            [item_index[item_id] for item_id in ranked_ids],
            dtype=np.int32,
        )

    query_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 5),
        min_df=2,
        max_features=120_000,
        sublinear_tf=True,
        dtype=np.float32,
    )
    history_matrix = query_vectorizer.fit_transform(query_table["_query_norm"])
    target_matrix = query_vectorizer.transform(targets["_query_norm"])

    neighbor_items: list[np.ndarray] = []
    predicted_microcats: list[list[int]] = []

    for start in range(0, len(targets), 32):
        scores = (target_matrix[start : start + 32] @ history_matrix.T).toarray()
        neighbors = np.argpartition(scores, -nearest_queries, axis=1)[:, -nearest_queries:]

        for row_scores, row_neighbors in zip(scores, neighbors):
            item_scores: defaultdict[int, float] = defaultdict(float)
            microcat_scores: defaultdict[int, float] = defaultdict(float)

            for neighbor in row_neighbors:
                similarity = float(row_scores[neighbor])
                if similarity <= 0:
                    continue
                query_text = query_table["_query_norm"].iloc[neighbor]
                microcat = int(query_table["dominant_microcat"].iloc[neighbor])
                microcat_scores[microcat] += similarity
                for candidate in item_history.get(query_text, ())[:20]:
                    item_scores[int(candidate)] += similarity

            neighbor_items.append(
                np.asarray(
                    sorted(item_scores, key=item_scores.get, reverse=True)[:RETRIEVAL_K],
                    dtype=np.int32,
                )
            )
            predicted_microcats.append(
                [
                    microcat
                    for microcat, _ in sorted(
                        microcat_scores.items(),
                        key=lambda pair: pair[1],
                        reverse=True,
                    )[:3]
                ]
            )

    location_counts = (
        history.groupby(["search_location_id", "item_location_id"])
        .size()
        .sort_values(ascending=False)
    )
    compatible_locations: dict[int, set[int]] = {}
    for search_location, counts in location_counts.groupby(level=0):
        compatible_locations[int(search_location)] = set(
            counts.index.get_level_values(1)[:8]
        )

    popularity_counts = history_in_corpus["item_id"].value_counts()
    popularity = np.log1p(
        np.asarray(
            [popularity_counts.get(item_id, 0) for item_id in benchmark_item_ids],
            dtype=np.float32,
        )
    )

    popular_candidates: list[np.ndarray] = []
    exact_history: list[np.ndarray] = []
    exact_history_by_location: list[np.ndarray] = []
    for row_number, row in targets.reset_index(drop=True).iterrows():
        compatible = compatible_locations.get(
            int(row["search_location_id"]),
            {int(row["search_location_id"])},
        )
        geo_mask = np.isin(item_locations, list(compatible))
        microcat_mask = np.isin(item_microcats, predicted_microcats[row_number])
        score = popularity + 0.25 * geo_mask + 0.15 * microcat_mask
        top = np.argpartition(score, -RETRIEVAL_K)[-RETRIEVAL_K:]
        popular_candidates.append(top[np.argsort(score[top])[::-1]].astype(np.int32))
        exact_history.append(
            item_history.get(row["_query_norm"], np.empty(0, dtype=np.int32))[:TOP_K]
        )
        exact_history_by_location.append(
            item_history_by_location.get(
                (str(row["_query_norm"]), int(row["search_location_id"])),
                np.empty(0, dtype=np.int32),
            )
        )

    return {
        "neighbor_items": neighbor_items,
        "predicted_microcats": predicted_microcats,
        "compatible_locations": compatible_locations,
        "popular_candidates": popular_candidates,
        "exact_history": exact_history,
        "exact_history_by_location": exact_history_by_location,
    }


validation_behavior = build_behavioral_sources(
    prior_train_validation,
    validation,
)
benchmark_behavior = build_behavioral_sources(
    train,
    benchmark_queries,
)

## 7. Объединение кандидатов

Вес полнотекстового словесного поиска принят за 1. Остальные источники имеют
меньшие веса и расширяют покрытие. Бонусы достаточно малы, чтобы не заменить
текстовую релевантность, но помогают выбрать 50 объектов из объединённого
пула примерно из 500-700 уникальных кандидатов

In [11]:
SOURCE_WEIGHTS = {
    "char_title": 0.50,
    "word_title": 0.25,
    "word_full": 1.00,
    "bm25_full": 1.15,
    "bm25_title_params": 0.40,
    "popularity": 0.10,
    "query_neighbors": 0.25,
    "exact_history_by_location": 1.75,
    "exact_history": 1.50,
}


def reciprocal_rank_fusion(
    target_queries: pd.DataFrame,
    char_source: np.ndarray,
    title_source: np.ndarray,
    full_source: np.ndarray,
    bm25_source: np.ndarray,
    bm25_params_source: np.ndarray,
    behavior: dict[str, object],
) -> list[np.ndarray]:
    """Объединяет источники и возвращает ровно TOP_K уникальных индексов."""
    predictions: list[np.ndarray] = []

    for i, row in target_queries.reset_index(drop=True).iterrows():
        sources = [
            char_source[i],
            title_source[i],
            full_source[i],
            bm25_source[i],
            bm25_params_source[i],
            behavior["popular_candidates"][i],
            behavior["neighbor_items"][i],
            behavior["exact_history_by_location"][i],
            behavior["exact_history"][i],
        ]
        weights = list(SOURCE_WEIGHTS.values())
        non_empty_sources = [source for source in sources if len(source)]
        candidates = np.unique(np.concatenate(non_empty_sources))
        positions = {candidate: position for position, candidate in enumerate(candidates)}
        scores = np.zeros(len(candidates), dtype=np.float32)

        for source, weight in zip(sources, weights):
            for rank, candidate in enumerate(source):
                scores[positions[int(candidate)]] += weight / (RRF_CONSTANT + rank)

        search_location = int(row["search_location_id"])
        compatible = behavior["compatible_locations"].get(
            search_location,
            {search_location},
        )
        scores += 0.03 * np.isin(item_locations[candidates], list(compatible))
        scores += 0.01 * (item_locations[candidates] == search_location)
        scores += 0.005 * np.isin(
            item_microcats[candidates],
            behavior["predicted_microcats"][i],
        )
        scores += 0.005 * item_quality[candidates]

        search_category = int(row["search_category"])
        if search_category != 0:
            scores += 0.04 * (item_categories[candidates] == search_category)

        top = candidates[np.argsort(scores)[::-1][:TOP_K]]
        predictions.append(top.astype(np.int32))

    return predictions


validation_count = len(validation)
validation_predictions = reciprocal_rank_fusion(
    validation,
    char_candidates[:validation_count],
    title_word_candidates[:validation_count],
    full_word_candidates[:validation_count],
    bm25_candidates[:validation_count],
    bm25_params_candidates[:validation_count],
    validation_behavior,
)

benchmark_predictions = reciprocal_rank_fusion(
    benchmark_queries,
    char_candidates[validation_count:],
    title_word_candidates[validation_count:],
    full_word_candidates[validation_count:],
    bm25_candidates[validation_count:],
    bm25_params_candidates[validation_count:],
    benchmark_behavior,
)

## 8. Результаты и анализ ошибок

In [12]:
def recall_at_50(predictions: list[np.ndarray], relevant: pd.Series) -> float:
    values = []
    for predicted_indices, relevant_ids in zip(predictions, relevant):
        predicted_ids = set(benchmark_item_ids[predicted_indices[:TOP_K]])
        values.append(len(predicted_ids & set(relevant_ids)) / len(relevant_ids))
    return float(np.mean(values))


validation_metrics = pd.Series(
    {
        "char TF-IDF, title": recall_at_50(
            list(char_candidates[:validation_count, :TOP_K]),
            validation["relevant_items"],
        ),
        "word TF-IDF, full text": recall_at_50(
            list(full_word_candidates[:validation_count, :TOP_K]),
            validation["relevant_items"],
        ),
        "BM25, full text": recall_at_50(
            list(bm25_candidates[:validation_count, :TOP_K]),
            validation["relevant_items"],
        ),
        "BM25, title and params": recall_at_50(
            list(bm25_params_candidates[:validation_count, :TOP_K]),
            validation["relevant_items"],
        ),
        "final hybrid": recall_at_50(
            validation_predictions,
            validation["relevant_items"],
        ),
    },
    name="Recall@50",
)
display(validation_metrics.to_frame().style.format("{:.4f}"))

,Recall@50
"char TF-IDF, title",0.2579
"word TF-IDF, full text",0.3300
"BM25, full text",0.3842
"BM25, title and params",0.2803
final hybrid,0.6703


In [13]:
item_location_by_id = benchmark_items.set_index("item_id")["item_location_id"]
item_title_by_id = benchmark_items.set_index("item_id")["item_title_raw"]

error_rows = []
for row_number, row in validation.iterrows():
    predicted_ids = set(benchmark_item_ids[validation_predictions[row_number]])
    relevant_ids = set(row["relevant_items"])
    exact_location_available = any(
        item_location_by_id.get(item_id) == row["search_location_id"]
        for item_id in relevant_ids
    )
    error_rows.append(
        {
            "query": row["search_query"],
            "hit": bool(predicted_ids & relevant_ids),
            "query_words": len(str(row["search_query"]).split()),
            "exact_location_positive": exact_location_available,
            "relevant_titles": " | ".join(
                str(item_title_by_id.get(item_id, "")) for item_id in relevant_ids
            ),
        }
    )

error_analysis = pd.DataFrame(error_rows)
display(
    error_analysis.groupby("exact_location_positive")["hit"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "hit_rate"})
)
display(error_analysis.loc[~error_analysis["hit"]].head(10))

,hit_rate,count
exact_location_positive,,
False,0.691176,136
True,0.665948,464


,query,hit,query_words,exact_location_positive,relevant_titles
0,обтяжка мебели,False,2,True,Ремонт и перетяжка мягкой мебели
7,газель межгород,False,2,True,Грузоперевозки переезды газель
8,сварка трубок кондиционера,False,3,True,Аргонная сварка услуги
14,ремонт газового оборудования авто,False,4,True,Оформление гбо. Внесение изменений в конструкцию
22,вывезу бесплатно ваш старый телевизор,False,5,False,Скупка техники б/у Выкуп
27,установка замка в дверь,False,4,True,Мастер на час
28,швея портной услуги,False,3,True,Швея/пошив одежды/костюмов/купальников/швейный...
30,школьный проект 9 класс,False,4,True,"Школьные презентации, проекты и плакаты, рефераты"
33,доставка пиломатериалов,False,2,False,Грузоперевозки газель 6 метров открытый борт
36,аренда фрезера по дереву,False,4,True,Аренда инструмента / прокат инструмента


Здесь особенно полезно смотреть не только на среднюю метрику, но и на
конкретные промахи. Например, запрос `поворот исполнения` ведёт к юридической
услуге. Без истории взаимодействий это почти невозможно понять из текста

Основные типы ошибок:

- редкие синонимы, которых нет ни в объявлениях, ни в похожих train-запросах
- очень узкие услуги, упомянутые только глубоко в длинном описании
- несовпадение уровней географии (город, район, область)
- неоднозначные короткие запросы, для которых 50 мест недостаточно покрывают
  несколько возможных интентов

Для них в гибрид добавлены символьный поиск, поведенческие соседи и мягкая,
а не жёсткая, география. Описание ограничено первыми 2500 символами как
компромисс между полнотой, памятью и временем выполнения

## 9. Формирование и строгая проверка `answer.csv`

In [14]:
answer = pd.DataFrame(
    {
        "query_id": benchmark_queries["query_id"],
        "answer": [
            " ".join(benchmark_item_ids[prediction])
            for prediction in benchmark_predictions
        ],
    }
)
answer.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")


def validate_submission(
    submission: pd.DataFrame,
    queries: pd.DataFrame,
    valid_item_ids: set[str],
) -> None:
    """Проверяет все формальные требования задания до отправки файла."""
    assert list(submission.columns) == ["query_id", "answer"]
    assert len(submission) == len(queries)
    assert submission["query_id"].is_unique
    assert set(submission["query_id"]) == set(queries["query_id"])
    assert submission["query_id"].str.fullmatch(r"[0-9A-Za-z]{16}").all()

    for item_string in submission["answer"]:
        item_ids = item_string.split()
        assert 1 <= len(item_ids) <= TOP_K
        assert len(item_ids) == len(set(item_ids))
        assert all(re.fullmatch(r"[0-9a-f]{16}", item_id) for item_id in item_ids)
        assert set(item_ids).issubset(valid_item_ids)


validate_submission(answer, benchmark_queries, benchmark_item_id_set)

print(f"Сохранено: {OUTPUT_PATH}")
print(f"Строк: {len(answer):,}; кандидатов в каждой строке: 1–{TOP_K}")
display(answer.head())

Сохранено: answer.csv
Строк: 2,452; кандидатов в каждой строке: 1–50


,query_id,answer
0,70DfDUpwjxB4lzFd,d01e14d2ffd4ddba 603623b4bd9e8f6c 355392014208...
1,JTrdTaZJvSiLPkXj,1c6d2079e07f4497 1344796085e92421 cbeccbecb1fb...
2,LZCZNoVG4AFUkVRJ,d8fce513e4f000a7 3f89b8062dc85f1c dab52187b450...
3,660ac9QVtXkRxZC3,bd2cb4500fad728e 4c15e7abba063ac5 976b75fa962b...
4,YgHcM9MVbxKnxD1e,d201edf373168f24 0f79695e8c342b3a c539e0661217...


## 10. Что можно улучшить

Если бы времени и вычислительных ресурсов было больше, я бы попробовал:

- русскую лемматизацию для словесных индексов
- локальную русскоязычную sentence-transformer модель
- явную иерархию «регион, город, район» вместо статистики совместимости
- обучаемую модель объединения кандидатов
- временное разбиение train, если бы в данных был timestamp

Текущая версия остаётся достаточно простой: она запускается на CPU и не
требует внешних сервисов или скачивания нейросетевых весов

## 11. Воспроизводимость

Решение детерминировано (`SEED = 42`) и не использует внешние API
Достаточно установить зависимости из `requirements.txt`, положить три
parquet-файла в `data/` и выполнить ноутбук сверху вниз. На Apple Silicon
основной расход времени приходится на словесный TF-IDF по полному тексту
вычисления укладываются в несколько минут и выполняются на CPU